# 第7章 函数、模块与代码组织

对应正文前置能力：把重复代码整理成可复用的函数，并理解后续 notebook 为什么会逐步过渡到模块化脚本。

这一章是 Part 0 里最重要的一章之一。后续 Part I/II 的很多代码，不是靠记住每一行完成的，而是靠理解函数如何接收输入、返回结果、封装细节，并把小任务组合成更大的工作流。

## 学习目标

- 理解函数参数、返回值、默认参数和关键字参数。
- 学会写最小可用的文档字符串和输入检查。
- 知道 `try` / `except` 在数据读取时为什么有用。
- 了解为什么把重复逻辑放进函数后，代码会更容易复用、测试和调试。
- 认识 `if __name__ == "__main__"` 这种模块入口写法。

## 先把读取逻辑封装起来

真实科研代码里，最常重复的动作通常是“读文件、检查字段、转换类型”。如果这些步骤每次都手写，很快就会变得难维护。先把它们放进函数里，后面的分析就只需要关心“我要分析什么”。

In [ ]:
import csv
import random
from pathlib import Path


def parse_float(value, field_name):
    """Convert a field to float with a clearer error message."""
    try:
        return float(value)
    except ValueError as exc:
        raise ValueError(f"字段 {field_name} 无法转换为浮点数: {value!r}") from exc


def load_planet_rows(path):
    """Load the small planetary demo table and convert numeric fields."""
    numeric_fields = ["semi_major_axis_au", "orbital_period_years", "mass_earth", "radius_earth"]
    with path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))

    for row_index, row in enumerate(rows, start=1):
        for field in numeric_fields:
            try:
                row[field] = parse_float(row[field], field)
            except ValueError as exc:
                raise ValueError(f"第 {row_index} 条记录的字段 {field} 出错") from exc
    return rows


demo_rows = load_planet_rows(Path("../../data/small/planetary_orbits_demo.csv"))
print("成功读取样本数:", len(demo_rows))
print("前三个行星:", [row["planet"] for row in demo_rows[:3]])

## 函数的输入和输出

函数最重要的事情只有两件：吃进去什么，吐出来什么。后续章节里，不管是统计、拟合还是分类，先把这件事说清楚，代码就会清楚很多。

In [ ]:
def mean_value(values):
    """Return the arithmetic mean of a non-empty sequence."""
    if not values:
        raise ValueError("values 不能为空")
    return sum(values) / len(values)


def summarize_numeric_field(rows, field):
    """Summarize a numeric field with count, mean, min, and max."""
    values = [row[field] for row in rows]
    return {
        "field": field,
        "count": len(values),
        "mean": mean_value(values),
        "minimum": min(values),
        "maximum": max(values),
    }


period_summary = summarize_numeric_field(demo_rows, "orbital_period_years")
mass_summary = summarize_numeric_field(demo_rows, "mass_earth")
print(period_summary)
print(mass_summary)

## 默认参数和关键字参数

默认参数让函数更容易用；关键字参数让调用更可读。后续很多科研函数都会用这种风格，因为它既短，又不容易让人误解。

In [ ]:
def filter_rows(rows, field, limit, keep_below=True):
    """Select rows by comparing a field against a threshold."""
    selected = []
    for row in rows:
        value = row[field]
        if keep_below and value <= limit:
            selected.append(row)
        elif not keep_below and value >= limit:
            selected.append(row)
    return selected


close_orbits = filter_rows(demo_rows, "semi_major_axis_au", 2.0)
giant_orbits = filter_rows(demo_rows, "mass_earth", 10.0, keep_below=False)
print("近轨道行星:", [row["planet"] for row in close_orbits])
print("大质量行星:", [row["planet"] for row in giant_orbits])

## 排序与返回值

很多函数不只返回一个数字，而是返回一段有结构的信息。后续章节里，`fit`、`predict`、`summary`、`evaluate` 之类的接口通常都遵循这个习惯。

In [ ]:
def top_n_rows(rows, field, n=3, reverse=True):
    """Return the top n rows sorted by a numeric field."""
    return sorted(rows, key=lambda row: row[field], reverse=reverse)[:n]


top_by_period = top_n_rows(demo_rows, "orbital_period_years", n=2)
top_by_mass = top_n_rows(demo_rows, "mass_earth", n=3)
print([row["planet"] for row in top_by_period])
print([row["planet"] for row in top_by_mass])

## 局部变量和作用域

函数内部的变量通常只在函数内部有效。这个习惯能减少命名冲突，也能让大型分析更容易分层。

In [ ]:
def period_and_mass_summary(rows):
    periods = [row["orbital_period_years"] for row in rows]
    masses = [row["mass_earth"] for row in rows]
    return {
        "mean_period": mean_value(periods),
        "mean_mass": mean_value(masses),
        "n_rows": len(rows),
    }


summary = period_and_mass_summary(demo_rows)
print(summary)
print("periods" in globals())

## 随机数与可复现性

后续章节会多次出现随机采样、初始化、打乱和模拟。只要涉及随机过程，就要把随机种子记录下来。这样别人才能重现你的结果，你自己也能复查。

In [ ]:
def simulate_measurements(values, sigma=0.05, seed=42):
    """Return a reproducible noisy copy of a sequence."""
    rng = random.Random(seed)
    return [rng.gauss(value, sigma) for value in values]


periods = [row["orbital_period_years"] for row in demo_rows[:4]]
first_run = simulate_measurements(periods, sigma=0.02, seed=7)
second_run = simulate_measurements(periods, sigma=0.02, seed=7)
third_run = simulate_measurements(periods, sigma=0.02, seed=8)
print([round(value, 3) for value in first_run])
print([round(value, 3) for value in second_run])
print([round(value, 3) for value in third_run])

## 模块入口的写法

当一段函数集合被搬进 `.py` 文件后，`if __name__ == "__main__":` 可以让这个文件既能被导入，也能被直接运行。很多后续脚本和工具都会采用这个约定。

In [ ]:
def run_demo():
    rows = load_planet_rows(Path("../../data/small/planetary_orbits_demo.csv"))
    summary = summarize_numeric_field(rows, "orbital_period_years")
    print("模块演示摘要:", summary)


if __name__ == "__main__":
    run_demo()

## 练习建议

1. 把 `filter_rows` 改写成接收一个比较函数，例如“是否大于阈值”。
2. 在 `load_planet_rows` 中加入对缺失字段的检查。
3. 写一个函数，返回“轨道周期最长的前三个行星”的名称列表。
4. 把 `simulate_measurements` 的结果改成返回“原值、扰动值”的成对列表。

## 小结

函数不是为了让代码看起来高级，而是为了让重复逻辑有一个稳定出口。只要你能把“输入是什么、输出是什么、错误怎么处理、默认值是什么”说清楚，就已经具备阅读后续大部分科研 Python 代码的基础了。